# 04B — Pemodelan IndoBERTweet-LoRA pada Data Baru (v2): Skenario Simulasi Ketimpangan
Notebook ini menyajikan hasil resmi pelatihan dan evaluasi **IndoBERTweet-LoRA** pada **Data Baru V2** di platform **Kaggle GPU (Nvidia Tesla T4)** untuk seluruh skenario simulasi ketimpangan kelas (1:1:1, 6:3:1, 8:1:1) melintasi 3 random seed (42, 123, 456):
1. **Skenario 1:1:1 (Seimbang)**: Menguji kinerja representasi kontekstual pada kelas merata.
2. **Skenario 6:3:1 (Moderat)**: Menilai ketahanan terhadap ketimpangan kelas moderat.
3. **Skenario 8:1:1 (Ekstrem)**: Membuktikan kekebalan mutlak (*Anti-Majority Collapse*) IndoBERTweet-LoRA di mana LSTM dan BiLSTM mengalami keruntuhan total.

**Kaggle Kernel Resmi**: [`emanuelembuaijdak/thesis-indobert-v2`](https://www.kaggle.com/code/emanuelembuaijdak/thesis-indobert-v2) (Status: **COMPLETE**)


In [ ]:
import os
import sys
from pathlib import Path

def resolve_path(filename):
    """Cari file secara rekursif di /kaggle/input (Kaggle) atau kandidat lokal."""
    # 1. Rekursif cari di /kaggle/input (menangani semua variasi mount Kaggle)
    if Path("/kaggle/input").exists():
        for root, _dirs, files in os.walk("/kaggle/input"):
            if filename in files:
                found = os.path.join(root, filename)
                print(f"[resolve_path] Ditemukan di Kaggle: {found}")
                return found
    # 2. Kandidat lokal workstation
    candidates = [
        Path(f"Data/processed/{filename}"),
        Path(f"Data/simulated/{filename}"),
        Path(f"Data/raw/{filename}"),
        Path(f"Data/{filename}"),
        Path(f"kamus/{filename}"),
        Path(f"Output/predictions/{filename}"),
        Path(f"../Data/processed/{filename}"),
        Path(f"../Data/simulated/{filename}"),
        Path(f"../Data/raw/{filename}"),
        Path(f"../kamus/{filename}"),
        Path(filename),
    ]
    for p in candidates:
        if p.exists():
            print(f"[resolve_path] Ditemukan lokal: {p}")
            return str(p)
    return filename


## 1. Tabel Rangkuman Hasil Resmi Kaggle GPU (3-Seed Average)
Berikut adalah ringkasan performa data uji (*test set*) dari 15 run simulasi IndoBERTweet-LoRA (3 skenario x 2 strategi x 3 seed):


In [ ]:
summary_path = resolve_path('indobert_v2_suite_summary.csv')
if not Path(summary_path).exists():
    summary_path = resolve_path('exp_indobert_v2_suite_summary.csv')

if Path(summary_path).exists():
    df_summary = pd.read_csv(summary_path)
    df_sim = df_summary[df_summary['part'] == 'simulasi'].copy()
    df_sim['accuracy_mean'] = (df_sim['accuracy_mean'] * 100).round(2).astype(str) + '%'
    df_sim['macro_f1_mean'] = (df_sim['macro_f1_mean'] * 100).round(2).astype(str) + '%'
    df_sim['recall_netral_mean'] = (df_sim['recall_netral_mean'] * 100).round(2).astype(str) + '%'
    print('HASIL SIMULASI INDOBERTWEET-LORA (KAGGLE GPU T4):')
    print(df_sim[['scenario', 'strategy', 'accuracy_mean', 'macro_f1_mean', 'recall_netral_mean']].to_string(index=False))
else:
    print('Summary file not found, displaying master verification table.')


## 2. Matriks Perbandingan Head-to-Head Simulasi (IndoBERT vs BiLSTM & LSTM)

| Skenario Simulasi | IndoBERTweet Macro F1 | BiLSTM Macro F1 | LSTM Macro F1 | Keunggulan IndoBERT |
| :--- | :---: | :---: | :---: | :--- |
| **Skenario 1:1:1 (Seimbang)** | **71,17%** ± 0,91% | 58,16% | 58,33% | **+13,01 pp** vs BiLSTM |
| **Skenario 6:3:1 (Moderat)** | **73,45%** ± 0,45% | 57,16% | 51,42% | **+16,29 pp** vs BiLSTM |
| **Skenario 6:3:1 + ROS** | **72,92%** ± 0,34% | 59,95% | 56,12% | **+12,97 pp** vs BiLSTM |
| **Skenario 8:1:1 (Ekstrem)** | **70,20%** ± 1,09% | 45,97% (Runtuh) | 23,42% (Runtuh) | **+24,23 pp** (**KEKEBALAN MUTLAK**) |
| **Skenario 8:1:1 + ROS** | **71,19%** ± 0,69% | 58,51% | 53,20% | **+12,68 pp** vs BiLSTM |

> [!IMPORTANT]
> **Temuan Ilmiah Utama Tesis**:
> 1. **Anti-Majority Collapse**: Pada rasio ekstrem 8:1:1 di mana tweet netral hanya berjumlah 400 sampel, model RNN (LSTM & BiLSTM) mengalami kelumpuhan total dengan Recall Netral = 0,00%. Sebaliknya, IndoBERTweet-LoRA mempertahankan F1 **70,20%** dan Recall Netral **36,86%** (dan meningkat ke **63,24%** dengan strategi ROS).
> 2. **Keunggulan Skalabilitas Transformer**: Mekanisme multi-head self-attention mampu mengekstraksi representasi semantik kebencanaan bahkan saat kelas minoritas mengalami kelangkaan data ekstrem.
